<img src="images/nn.jpg" width="450" height="300">

## Forward Propagation

<img src="images/weights.jpg" width="600" height="500">

<img src="images/hidden.jpg" width="600" height="500">

 Verilen 1 input $[x_1,x_2,x_3]$ için;

<img src="images/nn-1sample.jpg" width="300" height="150">


m input verisi için düşünürsek; 

for i = 1 to m: 

$ \quad z^{1(i)} = W^1 a^{0(i)} + b^1 $

$ \quad a^{1(i)} = \sigma(z^{1(i)}) $

$ \quad z^{2(i)} = W^2 a^{1(i)} + b^2 $

$ \quad a^{2(i)} = \sigma(z^{2(i)}) $

<img src="images/nn-all.jpg" width="600" height="300">

## Backpropagation

$$ \frac{\partial E}{\partial w_{jk}} = \frac{\partial E}{\partial a_{k}} \frac{\partial a_k}{\partial z_{k}} \frac{\partial z_k}{\partial w_{jk}} $$

## ANN Code

In [43]:
import sklearn.datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from matplotlib import pyplot
import numpy as np

def loadDataset():
    # LOAD DATASET
    iris = sklearn.datasets.load_iris()
    X = iris.data  #shape 150,4
    y = iris.target #shape 150,1
    
    # Preprocess Data - Adding bias term and labels to onehot encoding
    bias = np.ones((X.shape[0],1))  # shape 150,1 
    X = np.concatenate((bias,X), axis=1) # shape 150,5
    y = OneHotEncoding(y) # shape 150,3
    
    # Split data - X_train : 112,5  X_test : 38,5
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    
    return X_train, X_test, y_train, y_test

def OneHotEncoding(y):
    classes = np.unique(y, return_counts=False)
    encodedY = np.zeros((y.shape[0],classes.size))
    for i in range(y.shape[0]):
        encodedY[i][y[i]] = 1
    return encodedY

def networkInitialize(in_neurons, h_neurons, y_neurons): # parameters: 5,4,3
    weights_in = np.random.rand(h_neurons-1, in_neurons)
    weights_h = np.random.rand(y_neurons,h_neurons)
    weights = {"W_in":weights_in, "W_h":weights_h}
    
    '''
    bias_in = np.random.rand(h_neurons-1, in_neurons)
    bias_h = np.random.rand(y_neurons,h_neurons)
    biases = {"b_in":weights_in, "b_h":weights_h}
    '''
    
    return weights

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    return sigmoid(z)*(1 - sigmoid(z))

def forwardProp(X, weights):
    z1 = np.dot(weights["W_in"],X) # shape 3,112
    a1 = sigmoid(z1) # shape 3,112
    bias = np.ones((1,a1.shape[1])) # shape 1,112
    a1 = np.concatenate((bias,a1), axis=0) # shape 4,112

    z2 = np.dot(weights["W_h"],a1) # shape 3,112
    a2 = sigmoid(z2) # shape 3,112
    
    forwardParams = {"A_i":X, "Z_h":z1, "A_h":a1, "Z_o":z2, "predictions":a2}
    
    return forwardParams
    
def backProp(y_train, learning_rate, weights, forwardParams):
    sampleSize = y_train.shape[1] # 112
    dz2 = forwardParams["predictions"] - y_train # 3,112 - 3,112 
    dw2 = np.dot(dz2,forwardParams["A_h"].T)/sampleSize # shape 3,4
    
    dz1 = np.dot(weights["W_h"].T,dz2)*sigmoid_derivative(forwardParams["A_h"]) # shape 4,112
    dw1 = np.dot(dz1,forwardParams["A_i"].T)/sampleSize # shape 4,5
    
    # UPDATE WEIGHTS
    weights["W_in"] = weights["W_in"] - (learning_rate * dw1)
    weights["W_h"] = weights["W_h"] - (learning_rate * dw2)
    
    return weights
    
def evalualte(y_actual, y_predicted):
    tp =0
    tn =0
    
    for i in range(y_test.size):
        if y_actual[i] == 1 and y_predicted == 1:
            tp += 1
        elif y_actual[i] == 0 and y_predicted == 0:
            tn += 1

    acc = (tp + tn)/(y_actual.size)
    
    return acc
    
    
if __name__ == "__main__":
    X_train, X_test, y_train, y_test = loadDataset()
    # Adjust data to apply matrix multiplications
    X_train = X_train.T # shape 5,112
    X_test = X_test.T # shape 5,38
    y_train = y_train.T # shape 3,112
    y_test = y_test.T # shape 3,38    
    
    # Initialize network - neuron numbers including bias neuron
    weights = networkInitialize(X_train.shape[0], 4, y_train.shape[0])
    
    #for i in range(1000):
    forwardParams = forwardProp(X_train, weights)
    weights = backProp(y_train, 0.03, weights, forwardParams)
        
    forwardParams = forwardProp(X_train.T, weights)
    print(np.argmax(forwardParams["predictions"], axis =0))

(3, 4)
(4, 5)


ValueError: operands could not be broadcast together with shapes (3,5) (4,5) 